In [2]:
import numpy as np

a = np.array([5.0, 2.0, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5])
np.mean(a)

np.float64(1.1666666666666667)

In [3]:
3/20

0.15

In [8]:
from fastapi.testclient import TestClient
from api.main import app

def test_health_ok():
    client = TestClient(app)
    r = client.get("/health")
    assert r.status_code == 200
    assert r.json()["status"] == "ok"

def test_case_studies_lists_plants():
    client = TestClient(app)
    r = client.get("/case-studies")
    print(r.status_code)
    names = [item["name"] for item in r.json()]
    print("available case studies:", names)

test_case_studies_lists_plants()

200
available case studies: ['BallBeam', 'DCMotor', 'InvPendulum']


In [2]:
from fastapi.testclient import TestClient
from api.main import app

client = TestClient(app)

system_name = "ball_beam"   # <-- change to "dc_motor" or "inverted_pendulum" to explore others

body = {
    "system_name": system_name,
    "controller_type": "PID",
    "gains": {"Kp": 5.0, "Ki": 0.1, "Kd": 1.0},
    "scenario": {"initial_condition_range": [-0.5, 0.5], "randomness_level": 0.0, "disturbance_level": 0.0},
    "dt": 0.01,
    "max_time": 5.0,
}

r = client.post("/silo/simulate", json=body)
print("status code:", r.status_code)

data = r.json()
print("success:", data["success"])
print("metrics:", data["metrics"])
print("trajectory length:", len(data["trajectory"]))
print("first 10 trajectory points:", data["trajectory"][:10])
print("message:", data.get("message"))

status code: 200
success: True
metrics: {'mse': 0.011086437430127176, 'rmse': 0.10529215274714054, 'settling_time': 0.96, 'overshoot': 10.849443210767966, 'stable': True, 'stability_margin': 97.46772452238247, 'rise_time': 0.5, 'zero_crossings': 3, 'control_effort': 0.01281676176946765, 'control_zero_crossings': 8, 'ss_error': 0.002406275337956322}
trajectory length: 501
first 10 trajectory points: [-0.4822158974450761, -0.4822158974450761, -0.4817490725563501, -0.4808156743606573, -0.47939093515182457, -0.47745015113008804, -0.47496775718849876, -0.4719177955145514, -0.46827444382921585, -0.4640126791936559]
message: None


In [2]:
import time
from fastapi.testclient import TestClient
from api.main import app

client = TestClient(app)

start = client.post("/silo/start", json={
    "config": {"system_name": "ball_beam", "seed": 42, "max_scenarios": 2, "max_iter": 3, "controllers": ["PID"]},
    "control_objective": "stabilize ball position at 0",
})

print("status code:", start.status_code)

job_id = start.json()["job_id"]
print("job_id:", job_id)

# Poll until done (with a timeout so a stuck job can't hang forever)
deadline = time.time() + 30
status = None
while time.time() < deadline:
    s = client.get(f"/silo/{job_id}")
    status = s.json()["status"]
    # print("status:", s.json())
    # print("status:", s.json()["result_summary"])

    if status in ("completed", "failed"):
        break
    time.sleep(0.2)

# print("status:", s.json())
# print("status:", s.json()["result_summary"])

status code: 200
job_id: 09000c8b-4833-4c46-9193-d2a547b6d182

=== 🎛️ SELECTING CONTROLLER FOR SCENARIO 1 ===
Selected Controller: PID | Initial Params: {'Kp': 25.005, 'Ki': 25.005, 'Kd': 25.005, 'reasoning': 'Initial midpoint for PID'}

=== 🎭 DESIGNING SCENARIO LEVEL 1/2 ===
Scenario Scenario_1: IC Range [1.0, 1.0]
#1/3 | Type:PID | Kp:42.376 | Ki:30.190 | Kd:40.358 | MSE:484.9507 | Ts:inf | %OS:0.00 | Stable:False
=== 🔍 TERMINATOR DECISION: CONTINUE ===
#2/3 | Type:PID | Kp:30.930 | Ki:43.087 | Kd:28.872 | MSE:484.9507 | Ts:inf | %OS:0.00 | Stable:False
=== 🔍 TERMINATOR DECISION: TERMINATE_SUCCESS ===
✅ Scenario 1 completed successfully!

=== 🎛️ SELECTING CONTROLLER FOR SCENARIO 2 ===
Selected Controller: PID | Initial Params: {'Kp': 25.005, 'Ki': 25.005, 'Kd': 25.005, 'reasoning': 'Initial midpoint for PID'}

=== 🎭 DESIGNING SCENARIO LEVEL 2/2 ===
Scenario Scenario_2: IC Range [1.0, 1.0]
#1/3 | Type:PID | Kp:31.788 | Ki:18.248 | Kd:18.515 | MSE:484.9507 | Ts:inf | %OS:0.00 | Stable:

In [5]:
import statistics
from fastapi.testclient import TestClient
from api.main import app
import numpy as np

client = TestClient(app)

# def run_once(seed):
#     body = {"system_name": "ball_beam", "controller_type": "PID",
#             "gains": {"Kp": 5.0, "Ki": 0.1, "Kd": 1.0},
#             "scenario": {"initial_condition_range": [-0.5, 0.5], "randomness_level": 0.1, "disturbance_level": 0.0},
#             "dt": 0.01, "max_time": 5.0}
#     return client.post("/silo/simulate", json=body).json()["metrics"]["mse"]

def run_once(seed):
    np.random.seed(seed)          # <-- NOW the seed matters: it fixes the randomness for this run
    body = {"system_name": "ball_beam", "controller_type": "PID",
            "gains": {"Kp": 5.0, "Ki": 0.1, "Kd": 1.0},
            "scenario": {"initial_condition_range": [-0.5, 0.5], "randomness_level": 0.1, "disturbance_level": 0.0},
            "dt": 0.01, "max_time": 5.0}
    return client.post("/silo/simulate", json=body).json()["metrics"]["mse"]

def test_variance_across_seeds():
    values = [run_once(s) for s in range(5)]
    print("mse values across seeds:", values)
    # assert all(v >= 0 for v in values)          # mse is never negative
    spread = max(values) - min(values)
    print("spread across seeds:", spread)
    # assert spread < 1000                         # sane upper bound; tune after observing

test_variance_across_seeds()

mse values across seeds: [14.11677105005986, 13.344483886556691, 52.75207730748559, 0.740252951840229, 42.84047654082779]
spread across seeds: 52.011824355645366


In [6]:
# --- Setup / Test Data ---
CONTROLLER_GAINS = {
    "P": {"Kp": 5.0},
    "PI": {"Kp": 5.0, "Ki": 0.5},
    "PD": {"Kp": 5.0, "Kd": 1.0},
    "PID": {"Kp": 5.0, "Ki": 0.5, "Kd": 1.0},
}

SCENARIOS = {
    "calm": {
        "initial_condition_range": [-0.2, 0.2],
        "randomness_level": 0.0,
        "disturbance_level": 0.0,
    },
    "wide_start": {
        "initial_condition_range": [-2.0, 2.0],
        "randomness_level": 0.0,
        "disturbance_level": 0.0,
    },
    "noisy": {
        "initial_condition_range": [-0.5, 0.5],
        "randomness_level": 0.2,
        "disturbance_level": 0.0,
    },
    "disturbed": {
        "initial_condition_range": [-0.5, 0.5],
        "randomness_level": 0.0,
        "disturbance_level": 0.5,
    },
}


def _base_body(**overrides):
    """A default simulate request; override any field via keyword args."""
    body = {
        "system_name": "ball_beam",
        "controller_type": "PID",
        "gains": {"Kp": 5.0, "Ki": 0.1, "Kd": 1.0},
        "scenario": {
            "initial_condition_range": [-0.5, 0.5],
            "randomness_level": 0.0,
            "disturbance_level": 0.0,
        },
        "dt": 0.01,
        "max_time": 5.0,
    }
    body.update(overrides)
    return body


def _assert_ran_ok(response):
    """Assertions to ensure response status and contents are valid."""
    assert (
        response.status_code == 200
    ), f"HTTP Error status code: {response.status_code}"
    data = response.json()
    assert data.get("success") is True, "Expected success to be True"
    assert "mse" in data.get("metrics", {}), "'mse' missing from metrics"
    assert "stable" in data.get("metrics", {}), "'stable' missing from metrics"
    assert (
        len(data.get("trajectory", [])) > 0
    ), "Trajectory list is empty or missing"


# --- Helper Function to Run Test Cases ---
def run_test(test_name, test_func):
    try:
        test_func()
        print(f"  [PASS] {test_name}")
    except Exception as e:
        print(f"  [FAIL] {test_name} -> Error: {e}")


# --- Execution Cell Output ---
print("=== Running AutoTestSiLo Controller Variants ===")
for kind, gains in CONTROLLER_GAINS.items():

    def test():
        body = _base_body(controller_type=kind, gains=gains)
        _assert_ran_ok(client.post("/silo/simulate", json=body))

    run_test(f"Controller Kind: {kind}", test)

print("\n=== Running AutoTestSiLo Scenario Conditions ===")
for name, scenario in SCENARIOS.items():

    def test():
        body = _base_body(scenario=scenario)
        _assert_ran_ok(client.post("/silo/simulate", json=body))

    run_test(f"Scenario: {name}", test)

print("\n=== Running AutoTestSiLo Time Settings ===")
time_settings = [(0.01, 5.0), (0.005, 5.0), (0.02, 3.0), (0.01, 10.0)]
for dt, max_time in time_settings:

    def test():
        body = _base_body(dt=dt, max_time=max_time)
        r = client.post("/silo/simulate", json=body)
        _assert_ran_ok(r)
        assert (
            len(r.json()["trajectory"]) > 50
        ), "Trajectory length under threshold 50"

    run_test(f"Time Setting dt={dt}, max_time={max_time}", test)

=== Running AutoTestSiLo Controller Variants ===
  [PASS] Controller Kind: P
  [PASS] Controller Kind: PI
  [PASS] Controller Kind: PD
  [PASS] Controller Kind: PID

=== Running AutoTestSiLo Scenario Conditions ===
  [PASS] Scenario: calm
  [PASS] Scenario: wide_start
  [PASS] Scenario: noisy
  [PASS] Scenario: disturbed

=== Running AutoTestSiLo Time Settings ===
  [PASS] Time Setting dt=0.01, max_time=5.0
  [PASS] Time Setting dt=0.005, max_time=5.0
  [PASS] Time Setting dt=0.02, max_time=3.0
  [PASS] Time Setting dt=0.01, max_time=10.0
